# Predição de Gênero - Análise de Reviews

- **Aluno:** Pedro Yutaro Mont Morency Nakamura  
- **Data:** 17/05/2026
- **Objetivo:** Desenvolver um modelo de classificação para predizer gênero...

# Fase 2: Representação Semântica Densa e Algoritmos Não-Lineares

Este notebook implementa a **Fase 2** do projeto. O objetivo principal é abandonar o espaço esparso de altíssima dimensionalidade (N-gramas) e transitar para um espaço contínuo e denso de baixa dimensionalidade (300 dimensões) gerado por **Word Embeddings estáticos do spaCy**.

Como os modelos de agrupamento médio (*Mean Pooling*) sofrem de diluição semântica em textos curtos, adotaremos uma arquitetura **híbrida**. Injetaremos marcadores morfossintáticos (Q10), estruturais (Q4, Q6) e um Dicionário Discriminatório de alta precisão (Q9) para auxiliar as **Árvores de Decisão (HistGradientBoosting)** na inferência do gênero.

# 0. Configurações e Importações de Dependẽncias e Dados

In [1]:
import os
import time
import pickle
import pandas as pd
import numpy as np

# Visualização de Dados
import matplotlib.pyplot as plt
import seaborn as sns

# Pré-Processamento
import re
import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
import spacy
from tqdm import tqdm
from spacy.matcher import Matcher

# Estruturas e estatística
from scipy.stats import loguniform, randint, uniform, mannwhitneyu

# Classificadores e seleção
from xgboost import XGBClassifier
from sklearn.svm import SVC, LinearSVC
from sklearn.linear_model import LogisticRegression
from sklearn.feature_selection import RFE
from sklearn.preprocessing import StandardScaler, OneHotEncoder, LabelEncoder
from sklearn.neural_network import MLPClassifier
from sklearn.ensemble import HistGradientBoostingClassifier

# Validação e busca de hiperparâmetros
from sklearn.base import clone, BaseEstimator, TransformerMixin
from sklearn.model_selection import StratifiedKFold, RandomizedSearchCV
from sklearn.metrics import f1_score, classification_report, confusion_matrix, ConfusionMatrixDisplay
from sklearn.pipeline import Pipeline, FeatureUnion

## 0.1 Configuração de Gráficos

In [3]:
# Configurações de UI do Pandas
pd.set_option('display.max_columns', None)
sns.set_theme(style="whitegrid")

## 0.2 Carregamento do modelo SpaCy para Português

In [4]:
try:
    # Carrega apenas componentes necessarios para POS/lemmas.
    disable=["parser", "ner"]
    nlp = spacy.load("pt_core_news_lg", disable=disable)
    print("✅ SpaCy pt_core_news_lg carregado com sucesso!")
except Exception as e:
    print(f"❌ Erro ao carregar o modelo: {e}")

✅ SpaCy pt_core_news_lg carregado com sucesso!


## 0.3 Importação dos Datasets

In [7]:
# Caminhos para os dados
TRAIN_PATH = '../data/treino.csv'
VALIDATION_PATH = '../data/validacao.csv'

print("Carregando os datasets brutos...")
try:
    df_train_raw = pd.read_csv(TRAIN_PATH, encoding='utf-8', low_memory=False)
    df_val_raw = pd.read_csv(VALIDATION_PATH, encoding='utf-8', low_memory=False)
    print(f"Dataset de Treino carregado: {df_train_raw.shape[0]} linhas e {df_train_raw.shape[1]} colunas.")
    print(f"Dataset de Validação carregado: {df_val_raw.shape[0]} linhas and {df_val_raw.shape[1]} colunas.")
except FileNotFoundError as e:
    print(f"Erro: Certifique-se de que os arquivos 'treino.csv' e 'validacao.csv' estão no mesmo diretório. {e}")

Carregando os datasets brutos...
Dataset de Treino carregado: 76942 linhas e 14 colunas.
Dataset de Validação carregado: 25647 linhas and 14 colunas.


# 1. Tratamento Essencial e Engenharia de Atributos Inicial

Para garantir a consistência com a Análise Exploratória de Dados (EDA):
1. Unificamos as colunas de texto `review_title` e `review_text` em uma única feature chamada `full_text`.
2. Removemos registros cujo texto combinado resultou em strings vazias ou nulas, pois dados sem conteúdo textual não possuem sinal preditivo para NLP.
3. Isolamos as variáveis preditoras ($X$) do vetor de alvos binários ($y$).

In [8]:
def pre_process_pipeline(df):
    """
    Realiza o tratamento estrutural obrigatório: fusão de textos,
    limpeza de NaNs nas strings e descarte de linhas semanticamente vazias.
    """
    df_processed = df.copy()
    
    # Substitui nulos por strings vazias para evitar concatenação com NaN (que resultaria em NaN)
    df_processed['review_title'] = df_processed['review_title'].fillna("")
    df_processed['review_text'] = df_processed['review_text'].fillna("")
    
    # Criação da feature unificada validada na EDA
    df_processed['full_text'] = df_processed['review_title'] + " " + df_processed['review_text']
    df_processed['full_text'] = df_processed['full_text'].str.strip()
    
    # Remove as linhas onde o full_text ficou completamente sem conteúdo
    df_processed = df_processed[df_processed['full_text'].str.len() > 0]
    
    return df_processed

In [9]:
df_train = pre_process_pipeline(df_train_raw)
df_val = pre_process_pipeline(df_val_raw)
print("Aplicando tratamento estrutural Aplicado")

Aplicando tratamento estrutural Aplicado


Definição exata das colunas preditoras que serão consumidas pelo ColumnTransformer:

In [10]:
features = ['full_text', 'reviewer_birth_year', 'site_category_lv1', 'site_category_lv2']
target = 'reviewer_gender'

Separação definitiva de features e targets:

In [11]:
X_train = df_train[features].copy()
y_train = df_train[target].copy()

X_val = df_val[features].copy()
y_val = df_val[target].copy()

print("\n--- Estrutura Final das Matrizes para Modelagem ---")
print(f"X_train: {X_train.shape} | y_train: {y_train.shape}")
print(f"X_val:   {X_val.shape} | y_val:   {y_val.shape}")
print(f"Distribuição do Alvo no Treino:\n{y_train.value_counts(normalize=True).round(4) * 100}")


--- Estrutura Final das Matrizes para Modelagem ---
X_train: (76899, 4) | y_train: (76899,)
X_val:   (25635, 4) | y_val:   (25635,)
Distribuição do Alvo no Treino:
reviewer_gender
M    51.59
F    48.41
Name: proportion, dtype: float64


# 2. Injeção Dinâmica do Léxico Discriminatório (Q9)

Modelos densos baseados em média (*Mean Pooling*) não possuem o poder de observar palavras individuais isoladamente.

Para contornar essa falha de *design* do Word2Vec em resenhas curtas, extraímos as **Top 50 palavras mais polarizadas** (através do Log-Odds calculado na EDA) e as transformaremos em contagens de intersecção no extrator customizado.

## 2.1 Dicionários de Log-Odds Ratio (Extraídos do Treino na Q9)

O uso de Unigramas puros maximiza a intersecção eficiente com o doc do spaCy (Set() operation em O(1))

In [12]:
# Top 50 termos associados ao viés Masculino (Predominância em Hardware, Auto e Pragmatismo)
top_50_masc = ['esposa', 'grato', 'surpreso', 'arrependido', 'enganado', 'desapontado', 'namorada', 'frustrado', 'marchas', 'preocupado', 'indignado', 'barba', 'construído', 'recusa', 'sonoridade', 'satisfeito', 'lesado', 'barbear', 'bluray', 'leds', 'decepcionado', 'hz', 'martelo', 'construção', 'aparar', 'mb', 'ray', 'razer', 'ferrari', 'full', 'ressarcido', 'repetidor', 'sintoniza', 'pre', 'hardware', 'cddvd', 'rock', 'veiculo', 'vga', 'desconto', 'operação', 'roteador', 'graves', 'fujam', 'reembolsado', 'sandisk', 'surpreendido', 'bastasse', 'nylon', 'permitir']

# Top 50 termos associados ao viés Feminino (Predominância em Beleza, Enxoval e Qualificadores)
top_50_fem = ['fofo', 'frizz', 'cheirinho', 'satisfeita', 'insatisfeita', 'apaixonada', 'chateada', 'marido', 'obrigada', 'sedoso', 'decepcionada', 'indignada', 'esposo', 'desapontada', 'revoltada', 'cacheados', 'ondulado', 'satisfeitíssima', 'lesada', 'amigas', 'ameiiii', 'enganada', 'huggiessofttouch', 'theinsidersbrasil', 'amando', 'namorado', 'absorção', 'impressionada', 'amei', 'ameiii', 'arrependida', 'máscara', 'acostumada', 'ameii', 'encantada', 'preocupada', 'condicionador', 'oleoso', 'hidratação', 'hidrata', 'cachos', 'depilador', 'colcha', 'alisa', 'amar', 'gracinha', 'huggies', 'shampoo', 'progressiva', 'casinha']

# Convertendo as listas para formato string minúscula para garantir match perfeito
top_50_masc = [str(w).lower() for w in top_50_masc]
top_50_fem = [str(w).lower() for w in top_50_fem]

print(f"✓ Dicionários Semânticos carregados: {len(top_50_masc)} termos Masculinos e {len(top_50_fem)} termos Femininos.")

✓ Dicionários Semânticos carregados: 50 termos Masculinos e 50 termos Femininos.


## Fase 2: Representação Densa (Word Embeddings)

O artigo de Morais & Merschmann (2022) utilizou Word2Vec para alimentar a Rede Neural (Multilayer Perceptron) nas bases de Blogs e Opiniões.

Aqui será explorado Word2Vec/Embeddings, onde cada palavra se torna um vetor denso de significado.

- **Geração do Embedding: TF-IDF vs. pt_core_news_lg**

#### Plano:

- **1. Vetorização Densa:** Será usado o modelo grande do spaCy (pt_core_news_lg) para extrair os vetores reais de cada palavra e tirar a média da frase (Mean Pooling).
- **2. O Algoritmo:** Matrizes densas são ideais para Redes Neurais (MLP). O artigo de referência de Morais & Merschmann utilizou exatamente o MLP com embeddings para tentar prever o gênero.

# 3. Extrator Denso Customizado (Embeddings + Features Exploratórias)

A classe `DenseFeatureExtractor` foi desenhada para atuar como o coração da Fase 2. Ela extrai simultaneamente:
1. **Word Embeddings (300D):** Vetor médio da frase usando o `pt_core_news_lg` do spaCy.
2. **Features Morfossintáticas (Q10):** Proporções de classes gramaticais (NOUN, ADJ, VERB, PROPN).
3. **Features Estilométricas e Estruturais (Q4, Q6):** Uso de CAIXA ALTA, pontuações expressivas, letras repetidas e contagem de palavras.
4. **Dicionário Lexical (Q9):** Contagem de intersecções exatas com os termos discriminatórios calculados no `analysis.ipynb`.

*Nota de Engenharia:* A limpeza de URLs e menções ocorre internamente no `transform`, *após* a contagem de CAIXA ALTA, preservando a integridade do texto para o modelo de linguagem. O construtor `__init__` realiza apenas atribuições puras para permitir a clonagem durante o `RandomizedSearchCV`.

### Passo 1: Refatoração do SuperTransformer "FeatureExtractor":

O objetivo é fazer o spaCy trabalhar uma única vez. A classe será modificada para retornar não apenas as colunas de estilometria e POS, mas também as 300 colunas do vetor de Embeddings de cada texto.

In [13]:
class DenseFeatureExtractor(BaseEstimator, TransformerMixin):
    def __init__(self, male_terms=None, female_terms=None):
        # O __init__ no sklearn deve conter apenas atribuições.
        self.male_terms = male_terms
        self.female_terms = female_terms

    def fit(self, X, y=None):
        # Conversão para set() executada no fit para buscas em O(1)
        self.male_terms_set_ = set(self.male_terms) if self.male_terms is not None else set()
        self.female_terms_set_ = set(self.female_terms) if self.female_terms is not None else set()
        return self
        
    def transform(self, X, y=None):
        # 1. Tratamento seguro de entrada (suporta Series ou Listas)
        if isinstance(X, (pd.Series, pd.DataFrame)):
            raw_texts = X.fillna("").astype(str).tolist()
        else:
            raw_texts = [str(t) if t is not None else "" for t in X]
            
        features = []
        cleaned_texts = []
        upper_counts = []
        
        # 2. Pré-processamento Híbrido (Antes do spaCy)
        for text in raw_texts:
            # Conta as CAIXA ALTA antes de converter para minúscula ou limpar
            upper_counts.append(len(re.findall(r'\b[A-ZÀ-Ú]{2,}\b', text)))
            
            # Remove ruídos web que geram vetores inúteis
            t = re.sub(r'http\S+|www\.\S+', ' ', text)
            t = re.sub(r'@\w+', ' ', t)
            cleaned_texts.append(t)
        
        # 3. Processamento Vetorial em Lote
        for i, doc in enumerate(nlp.pipe(cleaned_texts, batch_size=500)):
            text_lower = doc.text.lower()
            total_tokens = len(doc) + 1
            
            # Morfossintaxe (Q10)
            counts = {'NOUN': 0, 'ADJ': 0, 'VERB': 0, 'PROPN': 0}
            for token in doc:
                if token.pos_ in counts:
                    counts[token.pos_] += 1
                    
            # Estilometria Numérica e Estrutura (Q4 e Q6)
            multipunct_count = len(re.findall(r'[!?\.]{2,}', text_lower))
            repeatchar_count = len(re.findall(r'([a-zãõáéíóúâêîôûàèìòùç])\1{2,}', text_lower))
            word_count = len([t for t in doc if not t.is_punct])
            
            # Dicionário Lexical (Q9)
            tokens_str = set([t.text.lower() for t in doc])
            m_count = len(tokens_str.intersection(self.male_terms_set_))
            f_count = len(tokens_str.intersection(self.female_terms_set_))
            
            # Word Embeddings (Mean Pooling)
            vec = doc.vector if doc.has_vector else np.zeros(nlp.vocab.vectors_length)
                
            # Dicionário de Características Estruturais
            feat_dict = {
                'noun_ratio': counts['NOUN'] / total_tokens,
                'adj_ratio': counts['ADJ'] / total_tokens,
                'verb_ratio': counts['VERB'] / total_tokens,
                'propn_ratio': counts['PROPN'] / total_tokens,
                'upper_ratio': upper_counts[i] / total_tokens,
                'multipunct_ratio': multipunct_count / total_tokens,
                'repeatchar_ratio': repeatchar_count / total_tokens,
                'word_count': word_count,
                'male_lexicon': m_count,
                'female_lexicon': f_count
            }
            
            # Expande os 300 números do Embedding como colunas individuais
            for j in range(len(vec)):
                feat_dict[f'emb_{j}'] = vec[j]
                
            features.append(feat_dict)
            
        return pd.DataFrame(features)

print("✓ Classe DenseFeatureExtractor instanciada e parametrizada.")

✓ Classe DenseFeatureExtractor instanciada e parametrizada.


# 4. Column Transformer e Correção de Alvo para MLP

Na Fase 2, unificamos o `DenseFeatureExtractor` com os metadados categóricos da base (com proteção de cardinalidade usando `min_frequency` para as marcas). 
Como o nosso primeiro teste utiliza Redes Neurais (`MLPClassifier`), precisamos binarizar os rótulos de `'M'` e `'F'` para `0` e `1`. O *Scikit-Learn* apresenta uma falha na métrica de *early_stopping* do MLP quando processa alvos do tipo *string*, gerando erros internos de `np.isnan`.

TESTE-1: Aplicando Min frequency

In [14]:
# === DEFINIÇÃO DOS PIPELINES (FASE 2) ===

# Branch 1: NLP Denso (Substitui todo o TF-IDF e FeatureUnion anteriores)
# O StandardScaler aqui vai padronizar tanto as proporções (0.0 a 1.0) 
# quanto os 300 valores do Word2Vec, deixando tudo perfeito para a Rede Neural.
dense_nlp_pipeline_t1 = Pipeline([
    ('extractor', DenseFeatureExtractor(
        male_terms=top_50_masc, 
        female_terms=top_50_fem
    )),
    ('scaler', StandardScaler())
])

# Branch 2: Metadado Numérico (Idade)
dense_age_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

# Branch 3: Metadados Categóricos (Categoria Nível 1 e Nível 2)
dense_cat_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='constant', fill_value='Desconhecido')),
    ('ohe', OneHotEncoder(handle_unknown='ignore', min_frequency=0.01))
])

O ColumnTransformer unirá apenas:
- as 300 colunas semânticas + features de NLP (POS, Caixa Alta, Word Count)
- Idade Normalizada
- Categorias (OneHot).

In [15]:
# === UNIÃO FINAL ===
dense_preprocessor = ColumnTransformer(
    transformers=[
        ('dense_nlp', dense_nlp_pipeline_t1, 'full_text'),
        ('age', dense_age_pipeline, ['reviewer_birth_year']),
        ('category', dense_cat_pipeline, ['site_category_lv1', 'site_category_lv2'])
    ],
    remainder='drop'
)

print("TESTE-1: DenseFeatureExtractor e ColumnTransformer da Fase 2 inicializados com sucesso!")

TESTE-1: DenseFeatureExtractor e ColumnTransformer da Fase 2 inicializados com sucesso!


#### Correção obrigatória de Alvo para a Rede Neural

In [16]:
le = LabelEncoder()
y_train_encoded = le.fit_transform(y_train)
y_val_encoded = le.transform(y_val)

print(f"✓ Mapeamento do Alvo: {dict(zip(le.classes_, le.transform(le.classes_)))}")
print("✓ ColumnTransformer 'dense_preprocessor' concluído.")

✓ Mapeamento do Alvo: {'F': np.int64(0), 'M': np.int64(1)}
✓ ColumnTransformer 'dense_preprocessor' concluído.


## 5. Teste 0: Multi-Layer Perceptron (MLP)

Neste bloco, submetemos a nossa representação híbrida (Embeddings + Q9 + Metadados) a uma Rede Neural. Utilizaremos uma amostragem de 50% dos dados para realizar uma busca hiperparamétrica em tempo hábil, procurando a topologia matemática ideal (tamanho das camadas ocultas e regularização) capaz de separar o espaço contínuo.

In [ ]:
# 1. Pipeline da Rede Neural
mlp_pipeline = Pipeline([
    ('preprocessor', dense_preprocessor),
    ('classifier', MLPClassifier(
        max_iter=300, 
        early_stopping=True, # Interr 'reviewer_birth_year', 'site_category_lv1', 'site_category_lv2'ompe o treino se não houver melhoria, evitando overfitting
        random_state=42
    ))
])

# 2. Amostragem de 50% para otimização ágil
X_train_sample = X_train.sample(frac=0.50, random_state=42)
y_train_sample_encoded = pd.Series(y_train_encoded, index=y_train.index).loc[X_train_sample.index]

### 5.1 Malha de Hiperparâmetros para MLP


In [18]:
param_dist_mlp = {
    # Topologias: (Simples), (Funil), (Profunda)
    'classifier__hidden_layer_sizes': [(100,), (128, 64), (64, 32, 16)], 
    'classifier__activation': ['relu', 'tanh'],               # Funções de ativação não lineares
    'classifier__alpha': loguniform(1e-4, 1e-1),        # Regularização L2 contínua
    'classifier__learning_rate_init': loguniform(1e-3, 5e-2)
}

In [19]:
# 4. Configuração da Busca
random_search_mlp = RandomizedSearchCV(
    estimator=mlp_pipeline,
    param_distributions=param_dist_mlp,
    n_iter=10,        # Número de arquiteturas sorteadas
    cv=3,
    scoring='f1_macro', # Tratamento de peso igualitário
    n_jobs=1,         # n_jobs=1 obrigatório para evitar colapso de memória entre workers do spaCy
    random_state=42,
    verbose=2
)

In [20]:
# 5. Execução
print("A iniciar o Tuning da Rede Neural Multicamadas (Amostra 50%)...")
start_time = time.time()

random_search_mlp.fit(X_train_sample, y_train_sample_encoded)

print(f"\n✓ Tuning concluído em {(time.time() - start_time)/60:.2f} minutos.")
print(f"🔥 Melhor F1-Score (Macro) na Validação Cruzada: {random_search_mlp.best_score_:.4f}")
print("\n⚙️ Melhores Parâmetros Encontrados:")
for param, value in random_search_mlp.best_params_.items():
    print(f" - {param}: {value}")

A iniciar o Tuning da Rede Neural Multicamadas (Amostra 50%)...
Fitting 3 folds for each of 10 candidates, totalling 30 fits
[CV] END classifier__activation=relu, classifier__alpha=0.024526126311336778, classifier__hidden_layer_sizes=(64, 32, 16), classifier__learning_rate_init=0.01752410111812814; total time= 1.3min
[CV] END classifier__activation=relu, classifier__alpha=0.024526126311336778, classifier__hidden_layer_sizes=(64, 32, 16), classifier__learning_rate_init=0.01752410111812814; total time=  59.5s
[CV] END classifier__activation=relu, classifier__alpha=0.024526126311336778, classifier__hidden_layer_sizes=(64, 32, 16), classifier__learning_rate_init=0.01752410111812814; total time= 1.0min
[CV] END classifier__activation=relu, classifier__alpha=0.006173770394704574, classifier__hidden_layer_sizes=(128, 64), classifier__learning_rate_init=0.0018408992080552514; total time= 1.1min
[CV] END classifier__activation=relu, classifier__alpha=0.006173770394704574, classifier__hidden_lay

In [21]:

print(f"\n✓ Tuning concluído em {(time.time() - start_time)/60:.2f} minutos.")
print(f"🔥 Melhor F1-Score (Macro) na Validação Cruzada: {random_search_mlp.best_score_:.4f}")
print("\n⚙️ Melhores Parâmetros Encontrados:")
for param, value in random_search_mlp.best_params_.items():
    print(f" - {param}: {value}")


✓ Tuning concluído em 46.45 minutos.
🔥 Melhor F1-Score (Macro) na Validação Cruzada: 0.6938

⚙️ Melhores Parâmetros Encontrados:
 - classifier__activation: relu
 - classifier__alpha: 0.000137832374550072
 - classifier__hidden_layer_sizes: (64, 32, 16)
 - classifier__learning_rate_init: 0.014316013583282224


#### Fase 2 - Teste 1 - Pipeline Base do Gradient Boost

In [219]:
# 2. Pipeline Base do Gradient Boosting
fase2_pipeline_t1 = Pipeline([
    ('preprocessor', dense_preprocessor),
    ('classifier', HistGradientBoostingClassifier(random_state=42))
])

Grade de Hiperparâmetros para Árvores (Gradient Boosting)

In [209]:
param_dist_gb = {
    'classifier__max_iter': [500, 1_000, 1_500],             # Número de árvores
    'classifier__learning_rate': [0.001, 0.01],      # Velocidade de aprendizado
    'classifier__max_depth': [3, None],           # Profundidade de cada árvore
    'classifier__l2_regularization': [0.0, 10.0]     # Combate ao overfitting
}

In [220]:
param_dist_gb_advanced = {
    # Número de árvores (equivalente ao n_estimators)
    'classifier__max_iter': randint(200, 1500), 
    
    # Velocidade de aprendizado (loguniform é ótimo para testar grandezas diferentes)
    'classifier__learning_rate': loguniform(1e-3, 0.2), 
    
    # Profundidade (3 a 12 camadas If/Else)
    'classifier__max_depth': randint(3, 12),
    
    # Folhas mínimas (controla overfitting)
    'classifier__min_samples_leaf': randint(10, 50),
    
    # Regularização L2 para punir pesos muito grandes nas árvores
    'classifier__l2_regularization': loguniform(1e-3, 10.0),
    
    # Subamostragem de colunas (equivalente ao colsample_bytree)
    # Pega de 50% a 100% das colunas em cada nó para evitar dependência de uma única feature
    'classifier__max_features': uniform(0.5, 0.5) 
}

Criando a Amostra de 50%

In [221]:
X_train_sample_gb = X_train_p2_t1.sample(frac=0.50, random_state=42)
y_train_sample_gb = y_train.loc[X_train_sample_gb.index] # Não precisa mais de LabelEncoder!
print("Criado amostra de 50% dos dados para o Tuning do Gradient Boosting")

Criado amostra de 50% dos dados para o Tuning do Gradient Boosting


Configuração da Busca

In [222]:
random_search_gb = RandomizedSearchCV(
    estimator=fase2_pipeline_t1,
    param_distributions=param_dist_gb_advanced,
    n_iter=20, 
    cv=3,
    scoring='f1_macro',
    random_state=42,
    verbose=2
)

#### Fase 2 - Teste 1 - Execução :

In [223]:
# 6. Execução
print("\nTESTE-1.6: Iniciando Busca de Hiperparâmetros (MACRO-Avançados) do Gradient Boosting...")
start_time = time.time()

random_search_gb.fit(X_train_sample_gb, y_train_sample_gb)

print(f"\nBusca concluída em {(time.time() - start_time)/60:.2f} minutos.")
print(f"🔥 Melhor F1-Score na Validação Cruzada (Amostra 50%): {random_search_gb.best_score_:.4f}")
print("\n⚙️ Melhores Hiperparâmetros:")
for param, value in random_search_gb.best_params_.items():
    print(f" - {param}: {value}")


TESTE-1.6: Iniciando Busca de Hiperparâmetros (MACRO-Avançados) do Gradient Boosting...
Fitting 3 folds for each of 20 candidates, totalling 60 fits
[CV] END classifier__l2_regularization=0.03148911647956861, classifier__learning_rate=0.1540359659501924, classifier__max_depth=10, classifier__max_features=0.7993292420985183, classifier__max_iter=321, classifier__min_samples_leaf=28; total time=  44.7s
[CV] END classifier__l2_regularization=0.03148911647956861, classifier__learning_rate=0.1540359659501924, classifier__max_depth=10, classifier__max_features=0.7993292420985183, classifier__max_iter=321, classifier__min_samples_leaf=28; total time=  44.3s
[CV] END classifier__l2_regularization=0.03148911647956861, classifier__learning_rate=0.1540359659501924, classifier__max_depth=10, classifier__max_features=0.7993292420985183, classifier__max_iter=321, classifier__min_samples_leaf=28; total time=  44.4s
[CV] END classifier__l2_regularization=0.002511306167739001, classifier__learning_rat

---
### Passo 2 | Teste 2

Aplicando os melhores hiperparametros no HistGradientBoostingClassifier

Atualizando o pipeline do Gradient Boosting com os melhores hiperparâmetros encontrados:

In [109]:
fase2_gb_campeao = Pipeline([
    ('preprocessor', dense_preprocessor),
    ('classifier', HistGradientBoostingClassifier(
        max_iter=500, 
        max_depth=None, 
        learning_rate=0.01, 
        l2_regularization=1.0,
        random_state=42
    ))
])

In [111]:
print("Treinando o Campeão Gradient Boosting com 100% dos dados...")
fase2_gb_campeao.fit(X_train_p2_t1, y_train)

# 2. Avaliação no Conjunto de Validação
y_pred_gb_oficial = fase2_gb_campeao.predict(X_val)


Treinando o Campeão Gradient Boosting com 100% dos dados...


KeyboardInterrupt: 

In [113]:
f1_gb_oficial = f1_score(y_val, y_pred_gb_oficial, average='weighted')

print(f"\n======================================")
print(f"🏆 F1 Score (Weighted) - Fase 2 Oficial (GB Trees): {f1_gb_oficial:.4f}")
print(f"======================================\n")
print(classification_report(y_val, y_pred_gb_oficial))


🏆 F1 Score (Weighted) - Fase 2 Oficial (GB Trees): 0.6842

              precision    recall  f1-score   support

           F       0.69      0.63      0.66     12414
           M       0.68      0.74      0.71     13233

    accuracy                           0.69     25647
   macro avg       0.69      0.68      0.68     25647
weighted avg       0.69      0.69      0.68     25647



---
FASE 2 - TESTE 0

## Passo 3: Construção da Rede MLPClassifier

O artigo de Morais utilizou o Multilayer Perceptron (MLP) com Word2Vec. Como teremos uma matriz densa de cerca de ~350 colunas contínuas, o MLP é a escolha ideal para encontrar relações não lineares entre a semântica e a estruturação gramatical.

Configurando o MLPClassifier no Scikit-Learn.
Uma arquitetura de funil (ex: hidden_layer_sizes=(128, 64)) com ativação relu e early_stopping=True (para evitar overfitting).

### Passo 3.1 - Otimização de Hiperparâmetros do MLP

#### Aplicando Label Encoding para os targets:

In [87]:
le = LabelEncoder()
y_train_num = le.fit_transform(y_train) # Transforma 'F' ==> 0 'M' ===> 1
y_val_num = le.transform(y_val)
print("Mapeamento das classes:", dict(zip(le.classes_, le.transform(le.classes_))))

Mapeamento das classes: {'F': np.int64(0), 'M': np.int64(1)}


Amostragem Estratégica (50% dos dados para otimizar rápido)

In [144]:
print("Criando amostra de 50% dos dados para o Tuning da Rede Neural...")
X_train_sample_mlp = X_train.sample(frac=0.50, random_state=42)

Criando amostra de 50% dos dados para o Tuning da Rede Neural...


Pegando os mesmos índices sorteados no X_train para filtrar o y numérico:

In [145]:
sample_indexes = X_train_sample_mlp.index
y_train_num_series = pd.Series(y_train_num, index=y_train.index)
y_train_sample_mlp_num = y_train_num_series.loc[sample_indexes]

Definição do Pipeline da Fase 2 (Denso):

In [2]:
fase2_pipeline = Pipeline([
    ('preprocessor', dense_preprocessor),
    ('classifier', MLPClassifier(max_iter=400, early_stopping=True, random_state=42))
])

NameError: name 'Pipeline' is not defined

Grade de Hiperparâmetros para a Rede Neural:

In [147]:
param_dist_mlp = {
    # Topologia da Rede (Neurônios por camada)
    'classifier__hidden_layer_sizes': [
        (100,),              # 1 Camada Oculta: Rápido e menos propenso a overfit
        (128, 64),           # 2 Camadas: Clássico formato de funil
        (50, 25),            # 2 Camadas menores
        (128, 64, 32)        # 3 Camadas: Rede Neural Profunda (Deep Learning)
    ],
    # Solvers: Otimizadores dos pesos da rede
    'classifier__solver': ['adam', 'sgd'],
    # Função de Ativação
    'classifier__activation': ['relu', 'tanh', 'logistic'],
    # Regularização L2 (Combate ao Overfitting)
    'classifier__alpha': [0.0001, 0.001, 0.01, 0.1],
    # Taxa de aprendizado inicial
    'classifier__learning_rate_init': [0.001, 0.005, 0.01]
}

Configuração da Busca:

In [148]:
random_search_mlp = RandomizedSearchCV(
    estimator=fase2_pipeline,
    param_distributions=param_dist_mlp,
    n_iter=15, 
    cv=3,
    scoring='f1_weighted',
    random_state=42,
    verbose=2
)

Execução da amostra de 50%:

In [94]:
print("\nIniciando Busca de Hiperparâmetros Expandida do MLP...")
start_time = time.time()

random_search_mlp.fit(X_train_sample_mlp, y_train_sample_mlp_num)


Iniciando Busca de Hiperparâmetros Expandida do MLP...


NameError: name 'random_search_mlp' is not defined

In [93]:
print(f"\nBusca concluída em {(time.time() - start_time)/60:.2f} minutos.")
print(f"🔥 Melhor F1-Score na Validação Cruzada (Amostra): {random_search_mlp.best_score_:.4f}")
print("\n⚙️ Melhor Arquitetura da Rede Neural Encontrada:")
for param, value in random_search_mlp.best_params_.items():
    print(f" - {param}: {value}")

NameError: name 'start_time' is not defined

### Discussão do Resultado da Fase 2 - Teste 0:
```
Busca concluída em 487.07 minutos.
🔥 Melhor F1-Score na Validação Cruzada (Amostra): 0.6886

⚙️ Melhor Arquitetura da Rede Neural Encontrada:
 - classifier__solver: adam
 - classifier__learning_rate_init: 0.001
 - classifier__hidden_layer_sizes: (128, 64)
 - classifier__alpha: 0.001
 - classifier__activation: logistic
```

Resultados nem um pouco satisfatórios. O f1 de 68.86% não supera nem o pior modelo da fase anterior.

##### Sobre as Features:
Com base nos Qs analisados na etapa de exploração analítica, podemos fazer a implementação de novas features:
- top_15_male_lexicon_count (dicionario latente da Q9)
- top_15_female_lexicon_count (dicionario latente da Q9)
- product_brand no pipeline categorico + min_frequency

##### Sobre o Classificador:
- Redes Neurais (MLPs) são notoriamente difíceis de tunar para dados tabulares híbridos (Embeddings contínuos + Categorias binárias).
- O Estado da Arte atual para esse tipo de matriz mista são os Ensembles baseados em Árvores de Decisão, como o Gradient Boosting ou Random Forest.
- As árvores não se importam com a escala dos dados e lidam perfeitamente com a junção de Word2Vec e metadados.

---

### Discussão do Resultado da Fase 2 - Teste 1:
```
Busca concluída em 25.09 minutos.
🔥 Melhor F1-Score na Validação Cruzada (Amostra 50%): 0.6837

⚙️ Melhores Hiperparâmetros:
 - classifier__max_iter: 500
 - classifier__max_depth: None
 - classifier__learning_rate: 0.01
 - classifier__l2_regularization: 1.0

```

Resultados nem um pouco satisfatórios. O f1 de 68.86% não supera nem o pior modelo da fase anterior.

##### Sobre as Features:
Com base nos Qs analisados na etapa de exploração analítica, podemos fazer a implementação de novas features:
- top_15_male_lexicon_count (dicionario latente da Q9)
- top_15_female_lexicon_count (dicionario latente da Q9)
- product_brand no pipeline categorico + min_frequency

##### Sobre o Classificador:
- Redes Neurais (MLPs) são notoriamente difíceis de tunar para dados tabulares híbridos (Embeddings contínuos + Categorias binárias).
- O Estado da Arte atual para esse tipo de matriz mista são os Ensembles baseados em Árvores de Decisão, como o Gradient Boosting ou Random Forest.
- As árvores não se importam com a escala dos dados e lidam perfeitamente com a junção de Word2Vec e metadados.

---

### Discussão do Resultado da Fase 2 - Teste 1.1:
```
Busca concluída em 28.69 minutos.
🔥 Melhor F1-Score na Validação Cruzada (Amostra 50%): 0.6840

⚙️ Melhores Hiperparâmetros:
 - classifier__max_iter: 1500
 - classifier__max_depth: None
 - classifier__learning_rate: 0.01
 - classifier__l2_regularization: 0.0
```

### Discussão do Resultado da Fase 2 - Teste 1.2:
```
Busca concluída em 52.36 minutos.
🔥 Melhor F1-Score na Validação Cruzada (Amostra 50%): 0.6849

⚙️ Melhores Hiperparâmetros:
 - classifier__l2_regularization: 2.752717392942941
 - classifier__learning_rate: 0.036762755221346345
 - classifier__max_depth: 11
 - classifier__max_features: 0.5325257964926398
 - classifier__max_iter: 587
 - classifier__min_samples_leaf: 34
```

## Fase 2 - Teste 1:
**HistGradientBoostingClassifier**

#### Resultados:

In [47]:
y_pred_mlp = fase2_pipeline.predict(X_val)
f1 = f1_score(y_val_num, y_pred_mlp, average='weighted')
print(f"\n======================================")
print(f"🏆 F1 Score (Weighted) - FASE 2 - Sub 0 - MLP RandomSearchCV-1: {f1:.4f}")
print(f"======================================\n")

print("Relatório de Classificação Detalhado:")
print(classification_report(y_val_num, y_pred_mlp))

NameError: name 'fase2_pipeline' is not defined

## 6. Validação e Métricas

In [ ]:
y_pred = model.predict(X_val_vec)
f1 = f1_score(y_val, y_pred, average='weighted')
print(f"F1 Score: {f1:.4f}")

## 7. Salvar Modelo

In [ ]:
def save_model(model, filename: str, path = './output/'):
  localefile = path + filename + '.pkl'
  pickle.dump(model, open(localefile, 'wb'))
  print("✓ Modelo salvo!")

## 8. Conclusões

- Melhor F1 Score alcançado: 0.XXXX
- Principais desafios: ...
- Possíveis melhorias: ...